# Lab 3 — MLflow: Datasets and Evaluations

---

> **🎓 TUTOR NOTES — Learning Goals**
>
> | # | Concept | What the student should understand |
> |---|---------|-------------------------------------|
> | 1 | **Creating a dataset** | Use `mlflow.data.from_pandas()` to create a dataset; log it with `mlflow.log_input()` |
> | 2 | **Adding entries** | Build or extend the DataFrame (e.g. concat rows), then create and log a new dataset version |
> | 3 | **Running evaluations** | Use `mlflow.models.evaluate()` to run metrics and log results to a run |
> | 4 | **Comparing runs** | Use `mlflow.search_runs()` to compare experiments and reuse datasets/evaluations in later labs |
>
> **Lab journey**: We use the loan application data and a simple XGBoost model. You will create an evaluation dataset in MLflow, add entries to it, run a formal evaluation, and compare runs — so you can reuse these patterns in the next sessions.

---

## Setup

In [8]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

import mlflow
import mlflow.xgboost
from mlflow.models import infer_signature

print('Imports OK')

Imports OK


In [9]:
df = pd.read_csv(Path('../data/loan_applications.csv'))

le = LabelEncoder()
df['loan_purpose_enc'] = le.fit_transform(df['loan_purpose'])

FEATURES = [
    'credit_score', 'annual_income', 'loan_amount',
    'num_defaults', 'employment_years', 'age', 'loan_purpose_enc'
]
TARGET = 'approved'

idx_train, idx_test = train_test_split(
    df.index, test_size=0.20, random_state=42, stratify=df[TARGET]
)

X_train = df.loc[idx_train, FEATURES]
X_test  = df.loc[idx_test, FEATURES]
y_train = df.loc[idx_train, TARGET]
y_test  = df.loc[idx_test, TARGET]

# Evaluation data: features + target (required by mlflow.models.evaluate)
eval_data = X_test.copy()
eval_data[TARGET] = y_test.values

print(f'Train: {len(y_train)} | Test: {len(y_test)}')
print(f'Approval rate — train: {y_train.mean():.1%}  test: {y_test.mean():.1%}')

Train: 800 | Test: 200
Approval rate — train: 65.0%  test: 65.0%


In [10]:
# Train a simple model (we'll log and evaluate it in the next sections)
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, verbosity=0)
model.fit(X_train, y_train)
print('Model trained')

Model trained


---
## Section 1 — Creating a Dataset in MLflow

So far we've kept evaluation data only in memory. **MLflow datasets** let you:
- Register a dataset (name, schema, digest) so runs can refer to it
- Log which data was used for training or evaluation via `mlflow.log_input()`
- Reuse the same dataset in later sessions and compare runs fairly

Create a dataset from a DataFrame with `mlflow.data.from_pandas()`, then log it to a run with `mlflow.log_input()`.

In [11]:
data_path = Path('../data/loan_applications.csv').resolve()

dataset = mlflow.data.from_pandas(
    eval_data,
    source=str(data_path),
    name='loan_eval_test',
    targets=TARGET,
)

print(f'Dataset name: {dataset.name}')
print(f'Dataset digest: {dataset.digest}')
print(f'Dataset schema: {dataset.schema}')
print(f'Rows: {len(eval_data)}')

Dataset name: loan_eval_test
Dataset digest: 00d6f2c7
Dataset schema: ['credit_score': long (required), 'annual_income': double (required), 'loan_amount': double (required), 'num_defaults': long (required), 'employment_years': double (required), 'age': long (required), 'loan_purpose_enc': long (required), 'approved': long (required)]
Rows: 200


/Users/matthias/Courses/ai_landscape/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/matthias/Courses/ai_landscape/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Intege

In [12]:
mlflow.set_experiment('lab03_mlflow_datasets_eval')

with mlflow.start_run(run_name='xgb_baseline'):
    mlflow.log_param('model', 'XGBClassifier')
    mlflow.log_param('n_features', len(FEATURES))

    # Log the evaluation dataset so this run is tied to the exact data used
    mlflow.log_input(dataset, context='evaluation')

    # Log model (needed for evaluate and for later reuse)
    signature = infer_signature(X_test, model.predict(X_test))
    model_info = mlflow.xgboost.log_model(model, artifact_path='model', signature=signature)

    # Run evaluation and log metrics/artifacts
    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data,
        targets=TARGET,
        model_type='classifier',
        evaluators=['default'],
    )

    # Log metrics from the evaluation result
    if result.metrics:
        mlflow.log_metrics(result.metrics)

print('Run complete. Metrics:', result.metrics)

/Users/matthias/Courses/ai_landscape/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/25 16:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 16:41:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will b

Run complete. Metrics: {'true_negatives': np.int64(57), 'false_positives': np.int64(13), 'false_negatives': np.int64(8), 'true_positives': np.int64(122), 'example_count': 200, 'accuracy_score': 0.895, 'recall_score': 0.9384615384615385, 'precision_score': 0.9037037037037037, 'f1_score': 0.9207547169811321, 'log_loss': 0.2143973050577532, 'roc_auc': 0.9698901098901099, 'precision_recall_auc': 0.9854037447937876}


---
## Section 2 — Adding Entries to a Dataset

Datasets in MLflow are **immutable snapshots**. To "add entries", you:
1. Build or extend your DataFrame (e.g. add more rows from another split or new data)
2. Create a new dataset with `from_pandas()` (optionally with a new name or version-like name)
3. Log it in a run with `log_input()` so that run is tied to the larger dataset

Example: we create a second dataset that includes both the original test set and an extra batch of rows (e.g. from training data), so we can evaluate on a larger set.

In [14]:
# "Add entries" by combining eval_data with another slice (e.g. a sample from train)
train_eval_slice = X_train.head(200).copy()
train_eval_slice[TARGET] = y_train.iloc[:200].values

eval_data_extended = pd.concat([eval_data, train_eval_slice], ignore_index=True)
print(f'Original eval rows: {len(eval_data)}')
print(f'After adding entries: {len(eval_data_extended)}')

Original eval rows: 200
After adding entries: 400


In [15]:
dataset_extended = mlflow.data.from_pandas(
    eval_data_extended,
    source=str(data_path),
    name='loan_eval_extended',
    targets=TARGET,
)

with mlflow.start_run(run_name='xgb_eval_extended'):
    mlflow.log_param('model', 'XGBClassifier')
    mlflow.log_param('n_features', len(FEATURES))
    mlflow.log_param('eval_rows', len(eval_data_extended))

    mlflow.log_input(dataset_extended, context='evaluation')

    signature = infer_signature(X_test, model.predict(X_test))
    model_info = mlflow.xgboost.log_model(model, artifact_path='model', signature=signature)

    result = mlflow.models.evaluate(
        model_info.model_uri,
        eval_data_extended,
        targets=TARGET,
        model_type='classifier',
        evaluators=['default'],
    )
    if result.metrics:
        mlflow.log_metrics(result.metrics)

print('Extended run complete. Metrics:', result.metrics)

/Users/matthias/Courses/ai_landscape/.venv/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/Users/matthias/Courses/ai_landscape/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Intege

Extended run complete. Metrics: {'true_negatives': np.int64(119), 'false_positives': np.int64(14), 'false_negatives': np.int64(8), 'true_positives': np.int64(259), 'example_count': 400, 'accuracy_score': 0.945, 'recall_score': 0.9700374531835206, 'precision_score': 0.9487179487179487, 'f1_score': 0.9592592592592593, 'log_loss': 0.13869935819848384, 'roc_auc': 0.9880600377347863, 'precision_recall_auc': 0.9942243775655577}


---
## Section 3 — Running Evaluations (Summary)

You've already used `mlflow.models.evaluate()` above. Recap:
- **model_uri**: from `mlflow.xgboost.log_model(..., artifact_path='model')` (or any logged model URI)
- **eval_data**: DataFrame that includes both feature columns and the **target** column
- **targets**: name of the target column
- **model_type**: `'classifier'` (or `'regressor'`)
- **evaluators**: `['default']` for built-in metrics (accuracy, F1, ROC-AUC, etc.)

The evaluation result contains metrics and optional artifacts (e.g. confusion matrix). Log them to the run with `mlflow.log_metrics(result.metrics)` so they appear in the UI and in `search_runs()`.

---
## Section 4 — Comparing Runs Programmatically

Use `mlflow.search_runs()` to compare all runs in the experiment — so you can pick the best model or dataset for the next sessions.

In [16]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name('lab03_mlflow_datasets_eval')
if experiment:
    runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    metric_cols = [c for c in runs_df.columns if c.startswith('metrics.')]
    display_cols = ['tags.mlflow.runName'] + metric_cols
    summary = (
        runs_df[display_cols]
        .rename(columns={
            'tags.mlflow.runName': 'Run',
            **{m: m.replace('metrics.', '') for m in metric_cols}
        })
        .set_index('Run')
        .dropna(axis=1, how='all')
        .round(4)
    )
    print(summary.to_string())
else:
    print('Experiment not found.')

print('\n💡 Open the MLflow UI: run `mlflow ui` in your terminal, then visit http://localhost:5000')

                   true_negatives  example_count  precision_recall_auc  log_loss  true_positives  false_positives  recall_score  f1_score  roc_auc  accuracy_score  false_negatives  precision_score
Run                                                                                                                                                                                                 
xgb_eval_extended           119.0          400.0                0.9942    0.1387           259.0             14.0        0.9700    0.9593   0.9881           0.945              8.0           0.9487
xgb_baseline                 57.0          200.0                0.9854    0.2144           122.0             13.0        0.9385    0.9208   0.9699           0.895              8.0           0.9037
xgb_baseline                 57.0          200.0                0.9854    0.2144           122.0             13.0        0.9385    0.9208   0.9699           0.895              8.0           0.9037

💡 Open the MLf

---\n\n## Where MLflow stores data & using the UI\n\n**Storage (on disk, not in memory)**\n\n- **Tracking store**: By default MLflow writes run metadata (params, metrics, tags) to a folder `./mlruns` in the directory from which you run your code. A small SQLite DB and files live there, so runs persist after you close the notebook.\n- **Artifact store**: Logged models and other artifacts are stored under `mlruns/<experiment_id>/<run_id>/artifacts/`.\n- **Datasets**: When you call `log_input(dataset, context=...)`, MLflow stores only **dataset metadata** (name, digest, schema, profile), not the full DataFrame. The actual data stays in memory unless you also save it as an artifact (e.g. CSV) in the run.\n\n**Using the UI**\n\nFrom the directory that contains the `mlruns` folder (usually your project root), run in a terminal:\n\n```bash\nmlflow ui\n```\n\nThen open **http://localhost:5000** in your browser. You can browse experiments (e.g. `lab03_mlflow_datasets_eval`), compare runs, view metrics and params, see logged dataset inputs, and download or inspect artifacts (e.g. the logged model).

---
## Key Takeaways

| Concept | What you learned |
|---|---|
| **Creating a dataset** | `mlflow.data.from_pandas(df, source=..., name=..., targets=...)` then `mlflow.log_input(dataset, context='evaluation')` |
| **Adding entries** | Extend the DataFrame (e.g. `pd.concat`), create a new dataset with `from_pandas`, log it in a new run |
| **Running evaluations** | `mlflow.models.evaluate(model_uri, eval_data, targets=..., model_type='classifier', evaluators=['default'])` and `mlflow.log_metrics(result.metrics)` |
| **Comparing runs** | `mlflow.search_runs(experiment_ids=[...])` to get a table of runs and metrics for use in later sessions |

**Next up**: Use these MLflow patterns (datasets + evaluations) in the next labs to compare models and track which data was used for each run.